# Commodity Return Forecasting — EDA & Backtest


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname("__file__"), ".."))

from src.data import load_data
from src.config import TARGETS
from src.features import engineer_base_features, engineer_target_features
from src.backtest import walkforward, print_results
from src.analysis import (
    plot_target_distributions, check_stationarity, plot_missing_values,
    plot_feature_target_correlations, plot_rolling_correlations,
    plot_feature_correlation_matrix, plot_cumulative_pnl,
    print_backtest_stats, plot_feature_importance,
)
from src.models import make_elasticnet


In [ ]:
df, feature_cols = load_data()
FEATURE_COLS = feature_cols  # alias for the analysis functions


## 1. EDA


In [ ]:
plot_target_distributions(df, TARGETS)
check_stationarity(df, TARGETS)


In [ ]:
plot_missing_values(df, FEATURE_COLS)


In [ ]:
plot_feature_target_correlations(df, TARGETS, FEATURE_COLS)
plot_rolling_correlations(df, TARGETS, FEATURE_COLS)
plot_feature_correlation_matrix(df, FEATURE_COLS)


## 2. Feature Engineering


In [ ]:
# base features are target-independent (z-scores, ranks, vol)
# safe to compute on full dataset
df_base, base_cols = engineer_base_features(df, feature_cols)
base_feature_set = feature_cols + base_cols
print(f"Total base features: {len(base_feature_set)}")


## 3. Walk-Forward Backtest


In [ ]:
# elasticnet baseline on raw features
for target in TARGETS:
    results, oos, _ = walkforward(
        df, feature_cols, target,
        model_factory=make_elasticnet,
        use_early_stopping=False,
    )
    print_results(results, target, "ElasticNet")


In [ ]:
# xgboost on raw features
for target in TARGETS:
    results, oos, _ = walkforward(df, feature_cols, target)
    print_results(results, target, "XGBoost (raw)")


In [ ]:
# xgboost on engineered features
# target-specific features (lags, momentum) depend on the horizon
for target in TARGETS:
    df_full, tgt_cols = engineer_target_features(df_base, feature_cols, target)
    full_features = base_feature_set + tgt_cols
    
    results, oos, model = walkforward(df_full, full_features, target)
    print_results(results, target, "XGBoost (engineered)")
    
    print_backtest_stats(oos, target)
    plot_cumulative_pnl(oos, target)
    plot_feature_importance(model, n_top=20, title=target)
